In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-03-08 08:20:59--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M   111MB/s    in 0.2s    

2026-03-08 08:21:00 (111 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code

In [ ]:
# Filter out users with < 200 ratings and books with < 100 ratings
user_rating_counts = df_ratings['user'].value_counts()
book_rating_counts = df_ratings['isbn'].value_counts()

valid_users = user_rating_counts[user_rating_counts >= 200].index
valid_books = book_rating_counts[book_rating_counts >= 100].index

df_ratings_filtered = df_ratings[
    (df_ratings['user'].isin(valid_users)) &
    (df_ratings['isbn'].isin(valid_books))
]

# Merge the filtered ratings with the book titles
df_merged = pd.merge(df_ratings_filtered, df_books, on='isbn')

# Clean up the data by dropping duplicate ratings for the same book by the same user
df_merged = df_merged.drop_duplicates(subset=['title', 'user'])

#  Pivot the data into a 2D matrix (Rows = Titles, Columns = Users, Values = Ratings)
df_pivot = df_merged.pivot(index='title', columns='user', values='rating').fillna(0)

#  Convert to a sparse matrix for efficient memory usage
matrix_books = csr_matrix(df_pivot.values)

#  Initialize and train the KNN model
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(matrix_books)

NearestNeighbors(algorithm='brute', metric='cosine')

In [ ]:
# function to return recommended books
def get_recommends(book = ""):
    # Grab the rating vector for the requested book and reshape it for the model
    book_vector = df_pivot.loc[book].values.reshape(1, -1)

    #  Query the KNN model for the 6 nearest neighbors
    # (We ask for 6 because the 1st result is always the book itself at a distance of 0)
    distances, indices = model_knn.kneighbors(book_vector, n_neighbors=6)

    # Create the inner list that will hold the [title, distance] pairs
    recommended_books_list = []

    # Loop backwards (from index 5 down to 1) to reverse the order
    for i in range(5, 0, -1):
        neighbor_title = df_pivot.index[indices[0][i]]
        neighbor_dist = distances[0][i]
        recommended_books_list.append([neighbor_title, neighbor_dist])

    #  Package it into the final expected structure: [original_book, [recommendations]]
    recommended_books = [book, recommended_books_list]

    return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge! 🎉🎉🎉🎉🎉
